> [Amanda L Richer, Kent A Riemondy, Lakotah Hardie, Jay R Hesselberth, **Simultaneous measurement of biochemical phenotypes and gene expression in single cells**, *Nucleic Acids Research*, (2020)](https://doi.org/10.1093/nar/gkaa240)

> **Dataset:** Cell line models targeted with CRISPR technology and UNG<sup>KO</sup> and RNASEH2C<sup>KO</sup> used for single cell RNA experssion experiments (haicut-seq) in 3 timepoints. 
> - **Cell line:** Hap1 UNG<sup>KO</sup> (HZGHC001531c012) and RNASEH2C<sup>KO</sup> (HZGHC004633c003) cells.


**Aim:** 

- [Single-cell network biology for resolving cellular heterogeneity in human diseases](https://doi.org/10.1038/s12276-020-00528-0)

- [SCENIC: single-cell regulatory network inference and clustering](https://www.nature.com/articles/nmeth.4463)
    - https://scenic.aertslab.org/
    - https://github.com/aertslab/pySCENIC

# Gene expression network

> [**Evaluating methods of inferring gene regulatory networks highlights their lack of performance for single cell gene expression data**](https://doi.org/10.1186/s12859-018-2217-z)

> 1. We found that single cell network inference methods do not necessarily have better performance than general methods, and even for the method that performed well for simulated data (SCODE), it did not have a higher prediction accuracy than other methods for the experimental single cell data. 
> 2. In addition, we found that when applied to simulated data without drop-out, **GENIE3** was the best performer amongst all general methods, while this did not generalize to the simulated ‘single cell’ data where drop-out was induced. 
> 3. From this study, we therefore conclude that for single cell data, either generated from experiments or simulations, these networks methods had consistently poor performance for reconstructing networks.
> 4. Lastly, the PCA analysis shows that SCODE had a distinct pattern of prediction, while even the presumably similar methods (e.g., ARACNE and CLR are considered to be similar) did not consistently cluster together in the plots, indicating that their network topology was not necessarily more similar to each other than the other methods.
> 5. **Reconstructing GRNs from gene expression data has been one of the most important topics in systems biology.** Therefore, evaluating the performance of existing algorithms and understanding their limitations when applied to a newer wave of data, such as single cell gene expression data, is extremely helpful to facilitate further development of new methodologies that are specific for single cells. Such models would further provide new opportunities to understand cell-to-cell heterogeneity, and other biologically interesting questions such as stem cell differentiation and cancer development.



> [**PySCNet: A tool for reconstructing and analyzing gene regulatory network from single-cell RNA-Seq data**](https://www.biorxiv.org/content/10.1101/2020.12.18.423482v1)

> PySCNet accommodates an extensive variety of GRN construction algorithms, graph based
techniques and a user-friendly interface to investigate regulatory relationships among genes and
transcription factors from single-cell RNA-seq data. Instead of following the mainstream analysis
on cell behaviour, PySCNet opens a window to systematically inspect gene-gene / TF
communication at single cell resolved level. 
https://github.com/MingBit/PySCNet


> ### `GENIE3`
> https://github.com/vahuynh/GENIE3

> [GENIE3 bioconductor](https://bioconductor.org/packages/release/bioc/html/GENIE3.html)

> [**dynGENIE3: dynamical GENIE3 for the inference of gene networks from time series expression data**
](https://www.nature.com/articles/s41598-018-21715-0)

> **The elucidation of gene regulatory networks is one of the major challenges of systems biology.** Measurements about genes that are exploited by network inference methods are typically available either in the form of steady-state expression vectors or time series expression data. In our previous work, we proposed the GENIE3 method that exploits variable importance scores derived from Random forests to identify the regulators of each target gene. This method provided state-of-the-art performance on several benchmark datasets, but it could however not specifically be applied to time series expression data. We propose here an adaptation of the GENIE3 method, called dynamical GENIE3 (dynGENIE3), for handling both time series and steady-state expression data. 
> https://github.com/vahuynh/dynGENIE3

# pySCENIC

> `pySCENIC` is a lightning-fast python implementation of the SCENIC pipeline (Single-Cell rEgulatory Network Inference and Clustering) which enables biologists to infer transcription factors, gene regulatory networks and cell types from single-cell RNA-seq data.
https://github.com/aertslab/pySCENIC/

<img src="https://scenic.aertslab.org/images/scenic_workflow_v2.png" width="700" align="center"/>

https://github.com/aertslab/SCENICprotocol/blob/master/notebooks/PBMC10k_SCENIC-protocol-CLI.ipynb

In [20]:
import os, glob, re, pickle
from functools import partial
from collections import OrderedDict
import operator as op
from cytoolz import compose

import pandas as pd
import seaborn as sns
import numpy as np
import scanpy as sc
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt

from pyscenic.export import export2loom, add_scenic_metadata
from pyscenic.utils import load_motifs
from pyscenic.transform import df2regulons
from pyscenic.aucell import aucell
from pyscenic.binarization import binarize
from pyscenic.rss import regulon_specificity_scores
from pyscenic.plotting import plot_binarization, plot_rss

from IPython.display import HTML, display

In [44]:
import loompy as lp

Write filtered `loom` file:

In [41]:
adata = sc.read_h5ad('preprocessing/mix.h5ad.gz')
f_loom_path_scenic = 'preprocessing/mix.loom'

In [45]:
# create basic row and column attributes for the loom file:
row_attrs = {
    "Gene": np.array(adata.var_names) ,
}
col_attrs = {
    "CellID": np.array(adata.obs_names) ,
    "nGene": np.array( np.sum(adata.X.transpose()>0 , axis=0)).flatten() ,
    "nUMI": np.array( np.sum(adata.X.transpose() , axis=0)).flatten() ,
}
lp.create( f_loom_path_scenic, adata.X.transpose(), row_attrs, col_attrs)


### Download resources

In [6]:
!mkdir SCENIC
!mkdir SCENIC/resources

_Download auxiliary files: list of TFs, ranking databases and motif annotations (for human, in this example):_

source: https://www.nature.com/articles/s41596-020-0336-2

In [49]:
ls SCENIC/resources/

hg38__refseq-r80__10kb_up_and_down_tss.mc9nr.feather
hs_hgnc_tfs.txt
motifs-v9-nr.hgnc-m0.001-o0.0.tbl


## SCENIC steps

In [31]:
RESOURCES_FOLDERNAME = "SCENIC/resources/"
RESULTS_FOLDERNAME = "SCENIC/results/"
FIGURES_FOLDERNAME = "SCENIC/figures/"

In [32]:
BASE_URL = "http://motifcollections.aertslab.org/v9/logos/"
COLUMN_NAME_LOGO = "MotifLogo"
COLUMN_NAME_MOTIF_ID = "MotifID"
COLUMN_NAME_TARGETS = "TargetGenes"

In [33]:
sc.settings.figdir = FIGURES_FOLDERNAME

In [34]:
# Set maximum number of jobs for Scanpy.
sc.settings.njobs = 4

Auxilliary functions.

In [35]:
def savesvg(fname: str, fig, folder: str=FIGURES_FOLDERNAME) -> None:
    """
    Save figure as vector-based SVG image format.
    """
    fig.tight_layout()
    fig.savefig(os.path.join(folder, fname), format='svg')

In [36]:
def display_logos(df: pd.DataFrame, top_target_genes: int = 3, base_url: str = BASE_URL):
    """
    :param df:
    :param base_url:
    """
    # Make sure the original dataframe is not altered.
    df = df.copy()
    
    # Add column with URLs to sequence logo.
    def create_url(motif_id):
        return '<img src="{}{}.png" style="max-height:124px;"></img>'.format(base_url, motif_id)
    df[("Enrichment", COLUMN_NAME_LOGO)] = list(map(create_url, df.index.get_level_values(COLUMN_NAME_MOTIF_ID)))
    
    # Truncate TargetGenes.
    def truncate(col_val):
        return sorted(col_val, key=op.itemgetter(1))[:top_target_genes]
    df[("Enrichment", COLUMN_NAME_TARGETS)] = list(map(truncate, df[("Enrichment", COLUMN_NAME_TARGETS)]))
    
    MAX_COL_WIDTH = pd.get_option('display.max_colwidth')
    pd.set_option('display.max_colwidth', -1)
    display(HTML(df.head().to_html(escape=False)))
    pd.set_option('display.max_colwidth', MAX_COL_WIDTH)

## STEP 1: Gene regulatory network inference, and generation of co-expression modules
### Phase Ia: GRN inference using the GRNBoost2 algorithm
For this step the CLI version of SCENIC is used. This step can be deployed on an High Performance Computing system. 

We use the counts matrix (without log transformation or further processing) from the loom file we wrote earlier. 

**Output:** List of adjacencies between a TF and its targets stored in ADJACENCIES_FNAME.

In [55]:
DATASET_ID = 'mix'

ADJACENCIES_FNAME = f'{RESULTS_FOLDERNAME}/{DATASET_ID}.adjacencies.tsv'
MOTIFS_FNAME = f'{RESULTS_FOLDERNAME}/{DATASET_ID}.motifs.csv'
REGULONS_DAT_FNAME = f'{RESULTS_FOLDERNAME}/{DATASET_ID}.regulons.dat'
AUCELL_MTX_FNAME = f'{RESULTS_FOLDERNAME}/{DATASET_ID}.auc.csv'
BIN_MTX_FNAME = f'{RESULTS_FOLDERNAME}/{DATASET_ID}.bin.csv'
THR_FNAME = f'{RESULTS_FOLDERNAME}/{DATASET_ID}.thresholds.csv'
ANNDATA_FNAME = f'{RESULTS_FOLDERNAME}/{DATASET_ID}.h5ad'
LOOM_FNAME = f'{RESULTS_FOLDERNAME}/{DATASET_ID}.loom'

In [56]:
# transcription factors
HUMAN_TFS_FNAME = f'{RESOURCES_FOLDERNAME}/hs_hgnc_tfs.txt'
# ranking databases
DB_FNAME = f'{RESOURCES_FOLDERNAME}/hg38__refseq-r80__10kb_up_and_down_tss.mc9nr.feather'
# motif databases
MOTIF_FNAME = f'{RESOURCES_FOLDERNAME}/motifs-v9-nr.hgnc-m0.001-o0.0.tbl'

In [58]:
mkdir SCENIC/results

In [59]:
!pyscenic grn {f_loom_path_scenic} {HUMAN_TFS_FNAME} -o {ADJACENCIES_FNAME} --num_workers 4

OMP: Info #270: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.

2021-06-10 22:57:18,937 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2021-06-10 22:57:20,030 - pyscenic.cli.pyscenic - INFO - Inferring regulatory networks.
OMP: Info #270: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #270: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #270: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #270: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
preparing dask client
parsing input
creating dask graph
4 partitions
computing dask graph
^C
not shutting down client, client was created externally
finished


Read in the adjacencies matrix:

In [56]:
adjacencies = pd.read_csv(ADJACENCIES_FNAME, index_col=False, sep='\t')

In [ ]:
adjacencies.head()

## STEP 2-3: Regulon prediction aka cisTarget from CLI

For this step the CLI version of SCENIC is used. This step can be deployed on an High Performance Computing system.

**Output:** List of adjacencies between a TF and its targets stored in MOTIFS_FNAME.

Here, we use the `--mask_dropouts` option, which affects how the correlation between TF and target genes is calculated during module creation. It is important to note that prior to pySCENIC v0.9.18, the default behavior was to mask dropouts, while in v0.9.18 and later, the correlation is performed using the entire set of cells (including those with zero expression). When using the `modules_from_adjacencies` function directly in python instead of via the command line, the `rho_mask_dropouts` option can be used to control this.

In [ ]:
!pyscenic ctx adj.tsv \
    {f_db_names} \
    --annotations_fname {f_motif_path} \
    --expression_mtx_fname {f_loom_path_scenic} \
    --output reg.csv \
    --mask_dropouts \
    --num_workers 20

In [37]:
df_motifs = load_motifs(MOTIFS_FNAME)

NameError: name 'MOTIFS_FNAME' is not defined

## STEP 4: Cellular enrichment (aka AUCell) from CLI
> It is important to check that most cells have a substantial fraction of expressed/detected genes in the calculation of the AUC. The following histogram gives an idea of the distribution and allows selection of an appropriate threshold. In this plot, a few thresholds are highlighted, with the number of genes selected shown in red text and the corresponding percentile in parentheses). See the relevant section in the R tutorial for more information.
By using the default setting for `--auc_threshold` of 0.05, we see that 1192 genes are selected for the rankings based on the plot below.

In [ ]:
nGenesDetectedPerCell = np.sum(adata.X>0, axis=1)
percentiles = nGenesDetectedPerCell.quantile([.01, .05, .10, .50, 1])
print(percentiles)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5), dpi=150)
sns.distplot(nGenesDetectedPerCell, norm_hist=False, kde=False, bins='fd')
for i,x in enumerate(percentiles):
    fig.gca().axvline(x=x, ymin=0,ymax=1, color='red')
    ax.text(x=x, y=ax.get_ylim()[1], s=f'{int(x)} ({percentiles.index.values[i]*100}%)', color='red', rotation=30, size='x-small',rotation_mode='anchor' )
ax.set_xlabel('# of genes')
ax.set_ylabel('# of cells')
fig.tight_layout()


In [ ]:
!pyscenic aucell \
    {f_loom_path_scenic} \
    reg.csv \
    --output {f_pyscenic_output} \
    --num_workers 20

## Visualization of SCENIC's AUC matrix

First, load the relevant data from the loom we just created

In [ ]:
# import json
# import zlib
# import base64

# # collect SCENIC AUCell output
# lf = lp.connect( f_pyscenic_output, mode='r+', validate=False )
# auc_mtx = pd.DataFrame( lf.ca.RegulonsAUC, index=lf.ca.CellID)
# lf.close()
# In [53]:
# import umap

# # UMAP
# runUmap = umap.UMAP(n_neighbors=10, min_dist=0.4, metric='correlation').fit_transform
# dr_umap = runUmap( auc_mtx )
# pd.DataFrame(dr_umap, columns=['X', 'Y'], index=auc_mtx.index).to_csv( "scenic_umap.txt", sep='\t')
# # tSNE
# tsne = TSNE( n_jobs=20 )
# dr_tsne = tsne.fit_transform( auc_mtx )
# pd.DataFrame(dr_tsne, columns=['X', 'Y'], index=auc_mtx.index).to_csv( "scenic_tsne.txt", sep='\t')
# Integrate the output
# Here, we combine the results from SCENIC and the Scanpy analysis into a SCope-compatible loom file

# In [57]:
# # scenic output
# lf = lp.connect( f_pyscenic_output, mode='r+', validate=False )
# meta = json.loads(zlib.decompress(base64.b64decode( lf.attrs.MetaData )))
# #exprMat = pd.DataFrame( lf[:,:], index=lf.ra.Gene, columns=lf.ca.CellID)
# auc_mtx = pd.DataFrame( lf.ca.RegulonsAUC, index=lf.ca.CellID)
# regulons = lf.ra.Regulons
# dr_umap = pd.read_csv( 'scenic_umap.txt', sep='\t', header=0, index_col=0 )
# dr_tsne = pd.read_csv( 'scenic_tsne.txt', sep='\t', header=0, index_col=0 )
# ###

Fix regulon objects to display properly in SCope:

In [ ]:
# auc_mtx.columns = auc_mtx.columns.str.replace('\(','_(')
# regulons.dtype.names = tuple( [ x.replace("(","_(") for x in regulons.dtype.names ] )
# # regulon thresholds
# rt = meta['regulonThresholds']
# for i,x in enumerate(rt):
#     tmp = x.get('regulon').replace("(","_(")
#     x.update( {'regulon': tmp} )
# Concatenate embeddings (tSNE, UMAP, etc.)

# In [60]:
# tsneDF = pd.DataFrame(adata.obsm['X_tsne'], columns=['_X', '_Y'])

# Embeddings_X = pd.DataFrame( index=lf.ca.CellID )
# Embeddings_X = pd.concat( [
#         pd.DataFrame(adata.obsm['X_umap'],index=adata.obs.index)[0] ,
#         pd.DataFrame(adata.obsm['X_pca'],index=adata.obs.index)[0] ,
#         dr_tsne['X'] ,
#         dr_umap['X']
#     ], sort=False, axis=1, join='outer' )
# Embeddings_X.columns = ['1','2','3','4']

# Embeddings_Y = pd.DataFrame( index=lf.ca.CellID )
# Embeddings_Y = pd.concat( [
#         pd.DataFrame(adata.obsm['X_umap'],index=adata.obs.index)[1] ,
#         pd.DataFrame(adata.obsm['X_pca'],index=adata.obs.index)[1] ,
#         dr_tsne['Y'] ,
#         dr_umap['Y']
#     ], sort=False, axis=1, join='outer' )
# Embeddings_Y.columns = ['1','2','3','4']

Metadata:

In [ ]:
# ### metadata
# metaJson = {}

# metaJson['embeddings'] = [
#     {
#         "id": -1,
#         "name": f"Scanpy t-SNE (highly variable genes)"
#     },
#     {
#         "id": 1,
#         "name": f"Scanpy UMAP  (highly variable genes)"
#     },
#     {
#         "id": 2,
#         "name": "Scanpy PC1/PC2"
#     },
#     {
#         "id": 3,
#         "name": "SCENIC AUC t-SNE"
#     },
#     {
#         "id": 4,
#         "name": "SCENIC AUC UMAP"
#     },
# ]

# metaJson["clusterings"] = [{
#             "id": 0,
#             "group": "Scanpy",
#             "name": "Scanpy louvain default resolution",
#             "clusters": [],
#         }]

# metaJson["metrics"] = [
#         {
#             "name": "nUMI"
#         }, {
#             "name": "nGene"
#         }, {
#             "name": "Percent_mito"
#         }
# ]

# metaJson["annotations"] = [
#     {
#         "name": "Louvain_clusters_Scanpy",
#         "values": list(set( adata.obs['louvain'].astype(np.str) ))
#     },
#     #{
#     #    "name": "Genotype",
#     #    "values": list(set(adata.obs['Genotype'].values))
#     #},
#     #{
#     #    "name": "Timepoint",
#     #    "values": list(set(adata.obs['Timepoint'].values))
#     #},
#     #{
#     #    "name": "Sample",
#     #    "values": list(set(adata.obs['Sample'].values))
#     #}
# ]

# # SCENIC regulon thresholds:
# metaJson["regulonThresholds"] = rt

# for i in range(max(set([int(x) for x in adata.obs['louvain']])) + 1):
#     clustDict = {}
#     clustDict['id'] = i
#     clustDict['description'] = f'Unannotated Cluster {i + 1}'
#     metaJson['clusterings'][0]['clusters'].append(clustDict)
    
# clusterings = pd.DataFrame()
# clusterings["0"] = adata.obs['louvain'].values.astype(np.int64)

Assemble loom file row and column attributes

In [ ]:
# def dfToNamedMatrix(df):
#     arr_ip = [tuple(i) for i in df.values]
#     dtyp = np.dtype(list(zip(df.dtypes.index, df.dtypes)))
#     arr = np.array(arr_ip, dtype=dtyp)
#     return arr

In [ ]:
# col_attrs = {
#     "CellID": np.array(adata.obs.index),
#     "nUMI": np.array(adata.obs['n_counts'].values),
#     "nGene": np.array(adata.obs['n_genes'].values),
#     "Louvain_clusters_Scanpy": np.array( adata.obs['louvain'].values ),
#     #"Genotype": np.array(adata.obs['Genotype'].values),
#     #"Timepoint": np.array(adata.obs['Timepoint'].values),
#     #"Sample": np.array(adata.obs['Sample'].values),
#     "Percent_mito": np.array(adata.obs['percent_mito'].values),
#     "Embedding": dfToNamedMatrix(tsneDF),
#     "Embeddings_X": dfToNamedMatrix(Embeddings_X),
#     "Embeddings_Y": dfToNamedMatrix(Embeddings_Y),
#     "RegulonsAUC": dfToNamedMatrix(auc_mtx),
#     "Clusterings": dfToNamedMatrix(clusterings),
#     "ClusterID": np.array(adata.obs['louvain'].values)
# }

# row_attrs = {
#     "Gene": lf.ra.Gene,
#     "Regulons": regulons,
# }

# attrs = {
#     "title": "sampleTitle",
#     "MetaData": json.dumps(metaJson),
#     "Genome": 'hg38',
#     "SCopeTreeL1": "",
#     "SCopeTreeL2": "",
#     "SCopeTreeL3": ""
# }

# # compress the metadata field:
# attrs['MetaData'] = base64.b64encode(zlib.compress(json.dumps(metaJson).encode('ascii'))).decode('ascii')

Create a new loom file, copying the expression matrix from the open loom connection:

In [ ]:
# lp.create(
#     filename = f_final_loom ,
#     layers=lf[:,:],
#     row_attrs=row_attrs, 
#     col_attrs=col_attrs, 
#     file_attrs=attrs
# )
# lf.close() # close original pyscenic loom file

This loom file can now be imported into [SCope](https://scope.aertslab.org/).

# dynGENIE3

- `TS_data` is a list of 3 arrays, each array corresponding to a time series experiment. The number of time points (rows) does not need to be the same in all the experiments, but the number of genes (columns) must be the same.
- `time_points` is a list of 3 vectors, containing the corresponding time points.
- `decay_rates` is a vector containing gene decay rates.
- `genes_names` is a list containing the names of the genes.

In [1]:
from dynGENIE3 import *

/Users/abearab/anaconda3/envs/sc-analysis/lib/python3.6/site-packages/sklearn/utils/deprecation.py:143: FutureWarning: The sklearn.tree.tree module is  deprecated in version 0.22 and will be removed in version 0.24. The corresponding classes / functions should instead be imported from sklearn.tree. Anything that cannot be imported from sklearn.tree is now part of the private API.
  warnings.warn(message, FutureWarning)


In [2]:
help(dynGENIE3)

Help on function dynGENIE3 in module dynGENIE3:

dynGENIE3(TS_data, time_points, alpha='from_data', SS_data=None, gene_names=None, regulators='all', tree_method='RF', K='sqrt', ntrees=1000, compute_quality_scores=False, save_models=False, nthreads=1)
    Computation of tree-based scores for all putative regulatory links.
    
    Parameters
    ----------
    
    TS_data: list of numpy arrays
        List of arrays, where each array contains the gene expression values of one time series experiment. Each row of an array corresponds to a time point and each column corresponds to a gene. The i-th column of each array must correspond to the same gene.
    
    time_points: list of one-dimensional numpy arrays
        List of n vectors, where n is the number of time series (i.e. the number of arrays in TS_data), containing the time points of the different time series. The i-th vector specifies the time points of the i-th time series of TS_data.
    
    alpha: either 'from_data', a positiv

In [3]:
help(dynGENIE3_predict_doubleKO)

Help on function dynGENIE3_predict_doubleKO in module dynGENIE3:

dynGENIE3_predict_doubleKO(expr_WT, treeEstimators, alpha, gene_names, regulators, KO1_gene, KO2_gene, nTimePoints, deltaT)
    Prediction of gene expressions in a double knockout experiment.
    
    Parameters
    ----------
    
    expr_WT: vector containing the gene expressions in the wild-type.
    
    treeEstimators: list of tree models, as returned by the function dynGENIE3(), where the i-th model is the model predicting the expression of the i-th gene. 
        The i-th model must correspond to the i-th gene in expr_WT.
    
    alpha: a positive number or a vector of positive numbers
        Specifies the degradation rate of the different gene expressions. 
        When alpha is a vector of positives, the i-th element of the vector must specify the degradation rate of the i-th gene.
        When alpha is a positive number, all the genes are assumed to have the same degradation rate.
    
    gene_names: list o

In [ ]:
(VIM, alphas, prediction_score, stability_score, treeEstimators) = dynGENIE3(TS_data,time_points)

`VIM` is an array containing the scores of the putative regulatory links. $VIM(i,j)$
is the weight of the link directed from the $i$-th gene to $j$-th gene.
- `alphas` is a vector in which the i-th element is the decay rate of the $i$-th gene. By default, the decay rate αi of gene i is estimated from its time series data, by assuming an exponential decay $e^{{−α}_it}$ between the highest and lowest observed expression values of gene i.
- `prediction_score` is an empty list by default.
- `stability_score` is an empty list by default.
- `treeEstimators` is an empty list by default.

https://github.com/reactome/reactome2py

https://reactome.github.io/reactome2py/utils.html